# Previsione Consegne Materie Prime - Approccio Cumulativo

**Nomi:** [Inserire nomi]

**ID Studenti:** [Inserire ID]

**Team Name Kaggle:** [Inserire team name]

---

Questo notebook implementa un modello di previsione per le consegne cumulative di materie prime.

**Obiettivo**: Prevedere il peso cumulativo di ogni `rm_id` dal 1 Gennaio alla data target.

**Metrica**: Quantile Loss 0.2 (penalizza maggiormente le sovrastime).

**Approccio INNOVATIVO**: Prediciamo direttamente il peso cumulativo, non il giornaliero!

## 1. Import Librerie e Caricamento Dati

In [35]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Caricamento dei dati
print("Caricamento dei dati...")
receivals_df = pd.read_csv('data/kernel/receivals.csv')
prediction_mapping_df = pd.read_csv('data/prediction_mapping.csv')

# Conversione delle date
receivals_df['date_arrival'] = pd.to_datetime(receivals_df['date_arrival'], errors='coerce', utc=True)
receivals_df = receivals_df.dropna(subset=['date_arrival'])
receivals_df['date'] = receivals_df['date_arrival'].dt.tz_localize(None)

print(f"Receivals shape: {receivals_df.shape}")
print(f"Date range: {receivals_df['date'].min()} to {receivals_df['date'].max()}")
print(f"Unique rm_ids: {receivals_df['rm_id'].nunique()}")

Caricamento dei dati...
Receivals shape: (122590, 11)
Date range: 2004-06-15 11:34:00 to 2024-12-19 13:36:00
Unique rm_ids: 203
Receivals shape: (122590, 11)
Date range: 2004-06-15 11:34:00 to 2024-12-19 13:36:00
Unique rm_ids: 203


## 2. Aggregazione Giornaliera e Calcolo Cumulativo Storico

**NOVITÀ**: Calcoliamo il peso cumulativo DALL'INIZIO DI OGNI ANNO per ogni rm_id.

In [36]:
# Aggregazione giornaliera
daily_receivals = receivals_df.groupby(['rm_id', receivals_df['date'].dt.date])['net_weight'].sum().reset_index()
daily_receivals.columns = ['rm_id', 'date', 'net_weight']
daily_receivals['date'] = pd.to_datetime(daily_receivals['date'])

# IMPORTANTE: Calcolo peso cumulativo dall'inizio di ogni anno
daily_receivals['year'] = daily_receivals['date'].dt.year
daily_receivals = daily_receivals.sort_values(['rm_id', 'date'])

# Cumsum resettato all'inizio di ogni anno per ogni rm_id
daily_receivals['cumulative_weight'] = daily_receivals.groupby(['rm_id', 'year'])['net_weight'].cumsum()

print(f"Daily receivals shape: {daily_receivals.shape}")
print(f"\nPrime 20 righe (mostra cumulativo):")
print(daily_receivals.head(20))

Daily receivals shape: (41933, 5)

Prime 20 righe (mostra cumulativo):
    rm_id       date  net_weight  year  cumulative_weight
0   342.0 2004-06-23     24940.0  2004            24940.0
1   343.0 2005-03-29     21760.0  2005            21760.0
2   345.0 2004-09-01     22780.0  2004            22780.0
3   346.0 2004-06-24       820.0  2004              820.0
4   346.0 2004-06-30     21260.0  2004            22080.0
5   346.0 2004-07-28      2880.0  2004            24960.0
6   347.0 2004-06-17     29805.0  2004            29805.0
7   347.0 2004-06-21     14920.0  2004            44725.0
8   347.0 2004-07-28     11200.0  2004            55925.0
9   347.0 2004-09-03     20220.0  2004            76145.0
10  348.0 2004-09-03     24220.0  2004            24220.0
11  348.0 2004-09-06     24400.0  2004            48620.0
12  353.0 2004-09-29     46340.0  2004            46340.0
13  353.0 2004-12-28     24190.0  2004            70530.0
14  354.0 2004-07-08     26222.0  2004            26222.0
1

## 3. Creazione del Master Table con Target Cumulativo

In [37]:
# Estrazione rm_ids unici e date range
unique_rm_ids = receivals_df['rm_id'].unique()
start_date = daily_receivals['date'].min()
end_date = pd.Timestamp('2024-12-31')
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

print(f"Creazione master table per {len(unique_rm_ids)} rm_ids e {len(date_range)} giorni...")

# Creazione master table
multi_index = pd.MultiIndex.from_product([unique_rm_ids, date_range], names=['rm_id', 'date'])
master_df = pd.DataFrame(index=multi_index).reset_index()

# Merge con le consegne effettive (incluso cumulative_weight)
master_df = pd.merge(
    master_df, 
    daily_receivals[['rm_id', 'date', 'net_weight', 'cumulative_weight']], 
    on=['rm_id', 'date'], 
    how='left'
)

# Riempi net_weight con 0 per giorni senza consegne
master_df['net_weight'] = master_df['net_weight'].fillna(0)

# IMPORTANTE: Ricalcola cumulative_weight per tutti i giorni (anche quelli senza consegne)
master_df['year'] = master_df['date'].dt.year
master_df = master_df.sort_values(['rm_id', 'date'])
master_df['cumulative_weight'] = master_df.groupby(['rm_id', 'year'])['net_weight'].cumsum()

print(f"Master table shape: {master_df.shape}")
print(f"\nVerifica target cumulativo per un rm_id:")
sample_rm = 365
print(master_df[master_df['rm_id'] == sample_rm].head(30)[['date', 'net_weight', 'cumulative_weight']])

Creazione master table per 204 rm_ids e 7505 giorni...
Master table shape: (1531020, 5)

Verifica target cumulativo per un rm_id:
         date  net_weight  cumulative_weight
0  2004-06-15     83784.0            83784.0
1  2004-06-16     40460.0           124244.0
2  2004-06-17    154251.0           278495.0
3  2004-06-18    204600.0           483095.0
4  2004-06-19         0.0           483095.0
5  2004-06-20         0.0           483095.0
6  2004-06-21    136600.0           619695.0
7  2004-06-22    132509.0           752204.0
8  2004-06-23     88160.0           840364.0
9  2004-06-24    137362.0           977726.0
10 2004-06-25    100558.0          1078284.0
11 2004-06-26         0.0          1078284.0
12 2004-06-27         0.0          1078284.0
13 2004-06-28     78080.0          1156364.0
14 2004-06-29     64232.0          1220596.0
15 2004-06-30    137160.0          1357756.0
16 2004-07-01    144304.0          1502060.0
17 2004-07-02    126900.0          1628960.0
18 2004-07-03  

## 4. Feature Engineering

Creiamo feature basate su:
- Temporali (giorno, mese, etc.)
- Lag del peso giornaliero (non cumulativo)
- Statistiche rm_id

In [38]:
print("Creazione feature temporali...")

# Feature temporali
master_df['month'] = master_df['date'].dt.month
master_df['day'] = master_df['date'].dt.day
master_df['dayofweek'] = master_df['date'].dt.dayofweek
master_df['dayofyear'] = master_df['date'].dt.dayofyear
master_df['weekofyear'] = master_df['date'].dt.isocalendar().week.astype(int)
master_df['quarter'] = master_df['date'].dt.quarter

# IMPORTANTE: days_since_year_start come feature chiave per il cumulativo
master_df['days_since_year_start'] = (master_df['date'] - pd.to_datetime(master_df['year'].astype(str) + '-01-01')).dt.days + 1

print("Creazione feature lag e rolling (sul peso giornaliero, non cumulativo)...")
from joblib import Parallel, delayed

def compute_features_for_rm(rm_data):
    """Calcola feature lag e rolling per un singolo rm_id"""
    rm_data = rm_data.sort_values('date').copy()
    
    # Feature Lag sul peso GIORNALIERO
    for lag in [7, 28, 56]:
        rm_data[f'lag_{lag}d'] = rm_data['net_weight'].shift(lag)
    
    # Feature Rolling sul peso GIORNALIERO
    for window in [7, 28]:
        rolling_obj = rm_data['net_weight'].shift(1).rolling(window, min_periods=1)
        rm_data[f'rolling_mean_{window}d'] = rolling_obj.mean()
        rm_data[f'rolling_sum_{window}d'] = rolling_obj.sum()
    
    # Feature: cumulativo fino a N giorni fa (per dare context al modello)
    rm_data['cumulative_7d_ago'] = rm_data['cumulative_weight'].shift(7)
    rm_data['cumulative_28d_ago'] = rm_data['cumulative_weight'].shift(28)
    
    return rm_data

# Parallelizzazione
results = Parallel(n_jobs=-1, verbose=1)(
    delayed(compute_features_for_rm)(group) 
    for _, group in master_df.groupby('rm_id')
)

master_df = pd.concat(results, ignore_index=True).sort_values(['rm_id', 'date'])

# Statistiche rm_id
print("Creazione statistiche aggregate per rm_id...")
rm_stats = master_df.groupby('rm_id')['net_weight'].agg([
    ('rm_mean', 'mean'),
    ('rm_std', 'std'),
    ('rm_median', 'median')
]).reset_index()
master_df = pd.merge(master_df, rm_stats, on='rm_id', how='left')

print(f"\nMaster table con feature: {master_df.shape}")
print(f"Colonne: {master_df.columns.tolist()}")

Creazione feature temporali...
Creazione feature lag e rolling (sul peso giornaliero, non cumulativo)...
Creazione feature lag e rolling (sul peso giornaliero, non cumulativo)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    2.0s finished
[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    2.0s finished


Creazione statistiche aggregate per rm_id...

Master table con feature: (1523515, 24)
Colonne: ['rm_id', 'date', 'net_weight', 'cumulative_weight', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'days_since_year_start', 'lag_7d', 'lag_28d', 'lag_56d', 'rolling_mean_7d', 'rolling_sum_7d', 'rolling_mean_28d', 'rolling_sum_28d', 'cumulative_7d_ago', 'cumulative_28d_ago', 'rm_mean', 'rm_std', 'rm_median']


## 5. Preparazione Dataset per Training

**TARGET**: `cumulative_weight` (peso cumulativo dall'inizio dell'anno)

In [39]:
# Definizione feature e target
feature_cols = [
    'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter',
    'days_since_year_start',  # CHIAVE per predire il cumulativo
    'lag_7d', 'lag_28d', 'lag_56d',
    'rolling_mean_7d', 'rolling_sum_7d',
    'rolling_mean_28d', 'rolling_sum_28d',
    # RIMOSSO: cumulative_7d_ago, cumulative_28d_ago (non hanno senso tra anni diversi)
    'rm_mean', 'rm_std', 'rm_median'
]
target_col = 'cumulative_weight'  # CAMBIATO: target è il cumulativo!

# Split temporale: train fino a fine 2023, validation su 2024
train_df = master_df[master_df['date'] < '2024-01-01'].copy()
val_df = master_df[master_df['date'] >= '2024-01-01'].copy()

# Rimozione righe con NaN nelle feature
train_df = train_df.dropna(subset=feature_cols + [target_col])
val_df = val_df.dropna(subset=feature_cols + [target_col])

# Separazione feature e target
X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_val = val_df[feature_cols]
y_val = val_df[target_col]

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"\nTarget distribution (training):")
print(y_train.describe())
print(f"\nTarget distribution (validation):")
print(y_val.describe())

Training set: (1437849, 17)
Validation set: (74298, 17)

Target distribution (training):
count    1.437849e+06
mean     1.944276e+05
std      1.556875e+06
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.387622e+07
Name: cumulative_weight, dtype: float64

Target distribution (validation):
count    7.429800e+04
mean     2.204121e+05
std      1.069550e+06
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.503073e+07
Name: cumulative_weight, dtype: float64


## 6. Training del Modello LightGBM

Addestriamo a predire direttamente il peso cumulativo.

In [40]:
print("Training del modello LightGBM per PESO CUMULATIVO...")

params = {
    'objective': 'regression',  # MAE loss (simmetrica)
    'metric': 'mae',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

print(f"Training completato. Best iteration: {model.best_iteration_}")

# Valutazione
val_pred = model.predict(X_val)
val_pred = np.clip(val_pred, 0, None)

# Calcolo quantile loss
errors = y_val - val_pred
quantile_loss = np.where(errors >= 0, 0.2 * errors, -0.8 * errors)
mean_quantile_loss = quantile_loss.mean()

print(f"\nValidation Quantile Loss (0.2): {mean_quantile_loss:.4f}")
print(f"Validation MAE: {np.abs(errors).mean():.2f}")
print(f"Validation RMSE: {np.sqrt((errors**2).mean()):.2f}")

Training del modello LightGBM per PESO CUMULATIVO...
Training completato. Best iteration: 215

Validation Quantile Loss (0.2): 51503.3835
Validation MAE: 106156.29
Validation RMSE: 611794.43
Training completato. Best iteration: 215

Validation Quantile Loss (0.2): 51503.3835
Validation MAE: 106156.29
Validation RMSE: 611794.43


## 7. Predizione per il 2025

Creiamo le predizioni cumulative per ogni data richiesta.

In [41]:
print("Creazione set di test per il 2025...")

# Estraiamo le date uniche richieste da prediction_mapping
prediction_mapping_df['forecast_end_date'] = pd.to_datetime(prediction_mapping_df['forecast_end_date'])
unique_forecast_dates = prediction_mapping_df['forecast_end_date'].unique()

print(f"Date uniche da predire: {len(unique_forecast_dates)}")
print(f"Range: {min(unique_forecast_dates)} to {max(unique_forecast_dates)}")

# Creiamo griglia per TUTTE le date dal 1 gen al 31 mag 2025 (per calcolare i lag)
test_date_range = pd.date_range(start='2025-01-01', end='2025-05-31', freq='D')
test_multi_index = pd.MultiIndex.from_product([unique_rm_ids, test_date_range], names=['rm_id', 'date'])
test_df = pd.DataFrame(index=test_multi_index).reset_index()

# Combiniamo con dati storici per calcolare lag
test_df['net_weight'] = np.nan
test_df['cumulative_weight'] = np.nan  # Sarà predetto
combined_df = pd.concat([master_df, test_df], ignore_index=True)
combined_df = combined_df.sort_values(['rm_id', 'date'])
combined_df['net_weight'] = combined_df['net_weight'].fillna(0)

print(f"\nCombined df shape: {combined_df.shape}")

Creazione set di test per il 2025...
Date uniche da predire: 150
Range: 2025-01-02 00:00:00 to 2025-05-31 00:00:00

Combined df shape: (1554319, 24)

Combined df shape: (1554319, 24)


## 8. Feature Engineering per Test Set

In [42]:
print("Creazione feature per il test set...")

# Feature temporali
combined_df['year'] = combined_df['date'].dt.year
combined_df['month'] = combined_df['date'].dt.month
combined_df['day'] = combined_df['date'].dt.day
combined_df['dayofweek'] = combined_df['date'].dt.dayofweek
combined_df['dayofyear'] = combined_df['date'].dt.dayofyear
combined_df['weekofyear'] = combined_df['date'].dt.isocalendar().week.astype(int)
combined_df['quarter'] = combined_df['date'].dt.quarter
combined_df['days_since_year_start'] = (combined_df['date'] - pd.to_datetime(combined_df['year'].astype(str) + '-01-01')).dt.days + 1

print("Calcolo feature lag e rolling per test set...")

def compute_test_features(rm_data):
    rm_data = rm_data.sort_values('date').copy()
    
    # Lag su net_weight (con dati storici reali fino a fine 2024)
    for lag in [7, 28, 56]:
        rm_data[f'lag_{lag}d'] = rm_data['net_weight'].shift(lag)
    
    for window in [7, 28]:
        rolling_obj = rm_data['net_weight'].shift(1).rolling(window, min_periods=1)
        rm_data[f'rolling_mean_{window}d'] = rolling_obj.mean()
        rm_data[f'rolling_sum_{window}d'] = rolling_obj.sum()
    
    # ATTENZIONE: cumulative_weight è NaN per 2025 (sarà predetto)
    # Quindi cumulative_7d_ago e cumulative_28d_ago potrebbero essere NaN all'inizio
    rm_data['cumulative_7d_ago'] = rm_data['cumulative_weight'].shift(7)
    rm_data['cumulative_28d_ago'] = rm_data['cumulative_weight'].shift(28)
    
    return rm_data

results = Parallel(n_jobs=-1, verbose=1)(
    delayed(compute_test_features)(group) 
    for _, group in combined_df.groupby('rm_id')
)

combined_df = pd.concat(results, ignore_index=True).sort_values(['rm_id', 'date'])

# Verifica che rm_stats esista e abbia le colonne corrette
print(f"rm_stats shape: {rm_stats.shape}")
print(f"rm_stats colonne: {rm_stats.columns.tolist()}")

# RIMUOVI le colonne rm_mean, rm_std, rm_median se già presenti (per evitare duplicati)
stats_cols = ['rm_mean', 'rm_std', 'rm_median']
existing_stats = [col for col in stats_cols if col in combined_df.columns]
if existing_stats:
    print(f"Rimozione colonne esistenti: {existing_stats}")
    combined_df = combined_df.drop(columns=existing_stats)

# Merge con rm_stats (assicurandosi che rm_id sia dello stesso tipo)
combined_df['rm_id'] = combined_df['rm_id'].astype(rm_stats['rm_id'].dtype)
combined_df = pd.merge(combined_df, rm_stats, on='rm_id', how='left')

print(f"combined_df colonne dopo il merge: {combined_df.columns.tolist()}")

# Verifica che le colonne statistiche siano presenti
missing_stats = [col for col in ['rm_mean', 'rm_std', 'rm_median'] if col not in combined_df.columns]
if missing_stats:
    print(f"ATTENZIONE: Colonne mancanti dopo merge: {missing_stats}")
else:
    print("Tutte le statistiche rm_id presenti!")

# Estrai solo 2025
final_test_df = combined_df[combined_df['year'] == 2025].copy()

# Gestione NaN
print(f"\nVerifica NaN nelle feature:")
for col in feature_cols:
    if col not in final_test_df.columns:
        print(f"  ERRORE: {col} non presente nel dataframe!")
        continue
    nan_count = final_test_df[col].isna().sum()
    if nan_count > 0:
        print(f"  {col}: {nan_count} NaN ({100*nan_count/len(final_test_df):.2f}%)")
        if 'cumulative' in col or 'lag' in col or 'rolling' in col:
            final_test_df[col] = final_test_df[col].fillna(0)

print(f"\nTest set 2025 shape: {final_test_df.shape}")
print(f"Colonne disponibili in final_test_df: {final_test_df.columns.tolist()}")

Creazione feature per il test set...
Calcolo feature lag e rolling per test set...
Calcolo feature lag e rolling per test set...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    0.7s finished
[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    0.7s finished


rm_stats shape: (203, 4)
rm_stats colonne: ['rm_id', 'rm_mean', 'rm_std', 'rm_median']
Rimozione colonne esistenti: ['rm_mean', 'rm_std', 'rm_median']
combined_df colonne dopo il merge: ['rm_id', 'date', 'net_weight', 'cumulative_weight', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'days_since_year_start', 'lag_7d', 'lag_28d', 'lag_56d', 'rolling_mean_7d', 'rolling_sum_7d', 'rolling_mean_28d', 'rolling_sum_28d', 'cumulative_7d_ago', 'cumulative_28d_ago', 'rm_mean', 'rm_std', 'rm_median']
Tutte le statistiche rm_id presenti!

Verifica NaN nelle feature:

Test set 2025 shape: (30653, 24)
Colonne disponibili in final_test_df: ['rm_id', 'date', 'net_weight', 'cumulative_weight', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'days_since_year_start', 'lag_7d', 'lag_28d', 'lag_56d', 'rolling_mean_7d', 'rolling_sum_7d', 'rolling_mean_28d', 'rolling_sum_28d', 'cumulative_7d_ago', 'cumulative_28d_ago', 'rm_mean', 'rm_std', 'rm_median']


## 9. Generazione Predizioni Cumulative

In [43]:
# DEBUG: Verifica feature importances
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Feature Importances:")
print(feature_importance.head(10))

# Verifica variabilità delle feature nel test set
print("\n\nVariabilità feature nel test set (per rm_id=365):")
sample_test = final_test_df[final_test_df['rm_id'] == 365][feature_cols].head(20)
for col in feature_cols:
    unique_vals = sample_test[col].nunique()
    print(f"  {col}: {unique_vals} valori unici (range: [{sample_test[col].min():.2f}, {sample_test[col].max():.2f}])")


Top 10 Feature Importances:
                  feature  importance
3               dayofyear        1333
12       rolling_mean_28d         983
14                rm_mean         809
10        rolling_mean_7d         612
15                 rm_std         579
9                 lag_56d         550
6   days_since_year_start         253
4              weekofyear         222
8                 lag_28d         209
1                     day         202


Variabilità feature nel test set (per rm_id=365):
  month: 1 valori unici (range: [1.00, 1.00])
  day: 20 valori unici (range: [1.00, 20.00])
  dayofweek: 7 valori unici (range: [0.00, 6.00])
  dayofyear: 20 valori unici (range: [1.00, 20.00])
  weekofyear: 4 valori unici (range: [1.00, 4.00])
  quarter: 1 valori unici (range: [1.00, 1.00])
  days_since_year_start: 20 valori unici (range: [1.00, 20.00])
  lag_7d: 1 valori unici (range: [0.00, 0.00])
  lag_28d: 1 valori unici (range: [0.00, 0.00])
  lag_56d: 1 valori unici (range: [0.00, 0.00])
  

In [44]:
print("Generazione predizioni CUMULATIVE per il 2025...")

X_test = final_test_df[feature_cols].copy()

# Predizione diretta del cumulativo
cumulative_predictions = model.predict(X_test)
cumulative_predictions = np.clip(cumulative_predictions, 0, None)

final_test_df['predicted_cumulative_weight'] = cumulative_predictions
final_test_df['rm_id'] = final_test_df['rm_id'].astype(int)

print(f"\nPredizioni PRIMA del fix monotono:")
print(f"Range: [{cumulative_predictions.min():.2f}, {cumulative_predictions.max():.2f}]")
print(f"Mean: {cumulative_predictions.mean():.2f}")

# IMPORTANTE: Fix per rendere le predizioni monotone crescenti per ogni rm_id
print("\nApplicazione vincolo MONOTONO CRESCENTE per rm_id...")
final_test_df = final_test_df.sort_values(['rm_id', 'date'])

def make_cumulative_monotonic(group):
    """Assicura che predicted_cumulative_weight sia monotono crescente"""
    group['predicted_cumulative_weight'] = group['predicted_cumulative_weight'].cummax()
    return group

final_test_df = final_test_df.groupby('rm_id', group_keys=False).apply(make_cumulative_monotonic)

# Ricalcola le statistiche
cumulative_predictions = final_test_df['predicted_cumulative_weight'].values

print(f"\nPredizioni DOPO il fix monotono:")
print(f"Predizioni generate: {len(cumulative_predictions)}")
print(f"Range: [{cumulative_predictions.min():.2f}, {cumulative_predictions.max():.2f}]")
print(f"Mean: {cumulative_predictions.mean():.2f}")
print(f"Predizioni > 0: {(cumulative_predictions > 0).sum()} ({100*(cumulative_predictions > 0).sum()/len(cumulative_predictions):.2f}%)")

# Esempio per un rm_id
sample_rm = 365
print(f"\nEsempio predizioni MONOTONE per rm_id={sample_rm}:")
sample_preds = final_test_df[final_test_df['rm_id'] == sample_rm][['date', 'days_since_year_start', 'predicted_cumulative_weight']].head(30)
print(sample_preds)


Generazione predizioni CUMULATIVE per il 2025...

Predizioni PRIMA del fix monotono:
Range: [0.00, 22008916.78]
Mean: 14197.02

Applicazione vincolo MONOTONO CRESCENTE per rm_id...

Predizioni DOPO il fix monotono:
Predizioni generate: 30653
Range: [0.00, 22008916.78]
Mean: 152385.87
Predizioni > 0: 28173 (91.91%)

Esempio predizioni MONOTONE per rm_id=365:
             date  days_since_year_start  predicted_cumulative_weight
114689 2025-01-01                      1                 32901.733706
114690 2025-01-02                      2                 32901.733706
114691 2025-01-03                      3                 33583.645219
114692 2025-01-04                      4                 34134.006080
114693 2025-01-05                      5                 34134.006080
114694 2025-01-06                      6                 34134.006080
114695 2025-01-07                      7                 34134.006080
114696 2025-01-08                      8                 34134.006080
114697 202

## 10. Creazione Submission File

In [45]:
print("Creazione file di submission...")

# Merge con prediction_mapping usando forecast_end_date
submission_df = prediction_mapping_df.copy()
submission_df = pd.merge(
    submission_df,
    final_test_df[['rm_id', 'date', 'predicted_cumulative_weight']],
    left_on=['rm_id', 'forecast_end_date'],
    right_on=['rm_id', 'date'],
    how='left'
)

# Gestione valori mancanti
submission_df['predicted_cumulative_weight'] = submission_df['predicted_cumulative_weight'].fillna(0)

# Formato finale
final_submission = submission_df[['ID', 'predicted_cumulative_weight']].copy()
final_submission.columns = ['ID', 'predicted_weight']

# Salvataggio
final_submission.to_csv('submission_cumulative.csv', index=False)

print(f"\nFile 'submission_cumulative.csv' creato con successo!")
print(f"Numero di predizioni: {len(final_submission)}")
print(f"\nStatistiche predizioni:")
print(final_submission['predicted_weight'].describe())
print(f"\nPredizioni > 0: {(final_submission['predicted_weight'] > 0).sum()} su {len(final_submission)}")
print(f"Percentuale non-zero: {100*(final_submission['predicted_weight'] > 0).sum()/len(final_submission):.2f}%")

Creazione file di submission...

File 'submission_cumulative.csv' creato con successo!
Numero di predizioni: 30450

Statistiche predizioni:
count    3.045000e+04
mean     1.533270e+05
std      1.477340e+06
min      0.000000e+00
25%      4.681700e+02
50%      1.229781e+03
75%      4.001206e+03
max      2.200892e+07
Name: predicted_weight, dtype: float64

Predizioni > 0: 28013 su 30450
Percentuale non-zero: 92.00%
